# 💰 Ultimate Mutual Fund Profit Predictor

## Advanced ML & Quantum Analysis for Maximum Profits

This notebook provides a comprehensive mutual fund analysis system using:
- **Advanced Feature Engineering**: Quantum scoring, momentum signals, smart beta factors
- **Stacked ML Ensemble**: Random Forest, Gradient Boosting, XGBoost, LightGBM
- **Market Regime Detection**: Adaptive strategy recommendations
- **Portfolio Optimization**: Modern portfolio theory implementation
- **Comprehensive Visualizations**: Interactive dashboards and insights

### Quick Start
1. Load your mutual fund data CSV
2. Run `maximize_profits(df)`
3. Get predictions, recommendations, and optimized portfolios

## 1. Setup and Imports

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Statistical & Optimization
from scipy import stats
from scipy.optimize import minimize
from datetime import datetime, timedelta

# Advanced ML (install if needed)
try:
    import xgboost as xgb
    print("✅ XGBoost loaded")
except ImportError:
    print("⚠️ XGBoost not available - install with: pip install xgboost")

try:
    import lightgbm as lgb
    print("✅ LightGBM loaded")
except ImportError:
    print("⚠️ LightGBM not available - install with: pip install lightgbm")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')

# Display Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.3f}'.format)

print("="*100)
print(" "*30 + "💰 ULTIMATE PROFIT PREDICTOR LOADED 💰")
print("="*100)

## 2. Advanced Feature Engineering

Creates quantum predictive features using financial engineering principles:
- Momentum indicators
- Risk-adjusted metrics
- Smart beta factors (Value, Quality, Size)
- Portfolio concentration measures
- Market regime adaptability scores

In [ ]:
def engineer_quantum_features(df):
    """Creates advanced predictive features using financial engineering"""
    print("\n🔬 ENGINEERING QUANTUM PREDICTIVE FEATURES...")
    
    df = df.copy()
    feature_count = 0
    
    # ============== MOMENTUM FEATURES ==============
    if 'Absolute Returns - 3M' in df.columns and 'Absolute Returns - 6M' in df.columns:
        # Price momentum with decay
        df['Momentum_3M_6M'] = df['Absolute Returns - 3M'] / (df['Absolute Returns - 6M'] + 0.01)
        df['Momentum_Strength'] = df['Absolute Returns - 3M'] * np.exp(-df['Volatility']/100)
        
        # Momentum quality (consistency)
        momentum_cols = ['Absolute Returns - 3M', 'Absolute Returns - 6M', 'Absolute Returns - 1Y']
        available_mom = [c for c in momentum_cols if c in df.columns]
        if len(available_mom) > 1:
            df['Momentum_Consistency'] = df[available_mom].std(axis=1) / (df[available_mom].mean(axis=1) + 0.01)
            df['Momentum_Trend'] = np.gradient(df[available_mom].values, axis=1).mean(axis=1)
        feature_count += 4
    
    # ============== RISK-ADJUSTED METRICS ==============
    if 'Sharpe Ratio' in df.columns:
        df['Enhanced_Sharpe'] = df['Sharpe Ratio'] * (1 + df.get('Alpha', 0)/100)
        df['Sharpe_Percentile'] = df['Sharpe Ratio'].rank(pct=True)
        feature_count += 2
    
    if 'Sortino Ratio' in df.columns and 'Sharpe Ratio' in df.columns:
        df['Downside_Protection_Ratio'] = df['Sortino Ratio'] / (df['Sharpe Ratio'] + 0.01)
        df['Risk_Asymmetry'] = df['Sortino Ratio'] - df['Sharpe Ratio']
        feature_count += 2
    
    # ============== ALPHA PERSISTENCE ==============
    if 'Alpha' in df.columns:
        df['Alpha_Squared'] = df['Alpha'] ** 2
        df['Alpha_Rank'] = df['Alpha'].rank(pct=True)
        df['Super_Alpha'] = (df['Alpha'] > df['Alpha'].quantile(0.9)).astype(int)
        feature_count += 3
    
    # ============== SMART BETA FACTORS ==============
    # Value Factor
    if 'PE Ratio' in df.columns and 'Category PE Ratio' in df.columns:
        df['Value_Factor'] = (df['Category PE Ratio'] - df['PE Ratio']) / df['Category PE Ratio']
        df['Deep_Value'] = (df['PE Ratio'] < df['Category PE Ratio'] * 0.7).astype(int)
        feature_count += 2
    
    # Quality Factor
    if '% Largecap Holding' in df.columns:
        df['Quality_Score'] = df['% Largecap Holding'] * 0.6
    if 'Maximum Drawdown' in df.columns:
        df['Drawdown_Control'] = 1 / (abs(df['Maximum Drawdown']) + 1)
    feature_count += 2
    
    # Size Factor
    if 'AUM' in df.columns:
        df['Size_Factor'] = np.log1p(df['AUM'])
        df['AUM_Growth_Potential'] = df['AUM'].rank(pct=True)
        df['Optimal_Size'] = ((df['AUM'] > df['AUM'].quantile(0.2)) & 
                              (df['AUM'] < df['AUM'].quantile(0.8))).astype(int)
        feature_count += 3
    
    # ============== PORTFOLIO CONCENTRATION ==============
    if '% Concentration - Top 3 Holdings' in df.columns:
        df['Diversification_Score'] = 100 - df['% Concentration - Top 3 Holdings']
        df['Over_Diversified'] = (df['% Concentration - Top 3 Holdings'] < 15).astype(int)
        df['Focused_Portfolio'] = (df['% Concentration - Top 3 Holdings'] > 30).astype(int)
        feature_count += 3
    
    # ============== EXPENSE EFFICIENCY ==============
    if 'Expense Ratio' in df.columns:
        df['Cost_Efficiency'] = 1 / (df['Expense Ratio'] + 0.01)
        df['Low_Cost_Advantage'] = (df['Expense Ratio'] < df['Expense Ratio'].quantile(0.25)).astype(int)
        
        if 'Alpha' in df.columns:
            df['Value_For_Money'] = df['Alpha'] / (df['Expense Ratio'] + 0.01)
        feature_count += 3
    
    # ============== CATEGORY LEADERSHIP ==============
    cat_outperform_cols = ['Returns vs sub-category - 1Y', 'Returns vs sub-category - 3Y', 
                           'Returns vs sub-category - 5Y', 'Returns vs sub-category - 10Y']
    available_cat = [c for c in cat_outperform_cols if c in df.columns]
    
    if available_cat:
        df['Category_Outperformance_Mean'] = df[available_cat].mean(axis=1)
        df['Category_Consistency'] = (df[available_cat] > 0).sum(axis=1) / len(available_cat)
        df['Category_Leader'] = (df['Category_Outperformance_Mean'] > 
                                 df['Category_Outperformance_Mean'].quantile(0.75)).astype(int)
        feature_count += 3
    
    # ============== REGIME ADAPTABILITY ==============
    if 'Volatility' in df.columns and 'Sharpe Ratio' in df.columns:
        df['All_Weather_Score'] = df['Sharpe Ratio'] / (df['Volatility'] + 1)
        df['Regime_Adaptability'] = df['Sharpe Ratio'] * np.exp(-df['Volatility']/50)
        feature_count += 2
    
    # ============== RECOVERY METRICS ==============
    if 'Maximum Drawdown' in df.columns and '% Away from ATH' in df.columns:
        df['Recovery_Speed'] = -df['% Away from ATH'] / (abs(df['Maximum Drawdown']) + 0.01)
        df['Near_ATH'] = (df['% Away from ATH'] > -5).astype(int)
        feature_count += 2
    
    # ============== COMPOSITE QUANTUM SCORE ==============
    quantum_features = []
    weights = {}
    
    if 'Enhanced_Sharpe' in df.columns:
        quantum_features.append('Enhanced_Sharpe')
        weights['Enhanced_Sharpe'] = 0.15
    
    if 'Alpha_Rank' in df.columns:
        quantum_features.append('Alpha_Rank')
        weights['Alpha_Rank'] = 0.15
    
    if 'Momentum_Strength' in df.columns:
        quantum_features.append('Momentum_Strength')
        weights['Momentum_Strength'] = 0.10
    
    if 'Category_Consistency' in df.columns:
        quantum_features.append('Category_Consistency')
        weights['Category_Consistency'] = 0.10
    
    if 'Cost_Efficiency' in df.columns:
        quantum_features.append('Cost_Efficiency')
        weights['Cost_Efficiency'] = 0.10
    
    if 'All_Weather_Score' in df.columns:
        quantum_features.append('All_Weather_Score')
        weights['All_Weather_Score'] = 0.10
    
    if 'Value_Factor' in df.columns:
        quantum_features.append('Value_Factor')
        weights['Value_Factor'] = 0.10
    
    if 'Diversification_Score' in df.columns:
        quantum_features.append('Diversification_Score')
        weights['Diversification_Score'] = 0.05
    
    if 'Recovery_Speed' in df.columns:
        quantum_features.append('Recovery_Speed')
        weights['Recovery_Speed'] = 0.05
    
    if 'Regime_Adaptability' in df.columns:
        quantum_features.append('Regime_Adaptability')
        weights['Regime_Adaptability'] = 0.10
    
    # Calculate Quantum Score
    if quantum_features:
        scaler = RobustScaler()
        normalized = pd.DataFrame(
            scaler.fit_transform(df[quantum_features].fillna(0)),
            columns=quantum_features,
            index=df.index
        )
        
        df['QUANTUM_SCORE'] = sum(
            normalized[feat] * weights.get(feat, 0.1) for feat in quantum_features
        )
        
        df['QUANTUM_SCORE'] = MinMaxScaler(feature_range=(0, 100)).fit_transform(
            df[['QUANTUM_SCORE']]
        ).flatten()
        
        df['QUANTUM_TIER'] = pd.qcut(df['QUANTUM_SCORE'], q=5, 
                                      labels=['Poor', 'Below Avg', 'Average', 'Good', 'Excellent'])
        feature_count += 2
    
    print(f"✅ Created {feature_count} quantum predictive features")
    
    return df

## 3. Stacked ML Ensemble

Builds a stacked ensemble combining:
- Random Forest
- Gradient Boosting
- Extra Trees
- XGBoost (if available)
- LightGBM (if available)

Uses weighted averaging based on individual model performance.

In [ ]:
def build_stacked_ml_predictor(df, target_horizon='1Y'):
    """Builds advanced stacked ML ensemble for superior predictions"""
    
    print(f"\n🤖 BUILDING STACKED ML ENSEMBLE FOR {target_horizon}...")
    
    base_features = [
        'Sharpe Ratio', 'Sortino Ratio', 'Alpha', 'Volatility', 'Maximum Drawdown',
        'Expense Ratio', '3Y Avg Annual Rolling Return'
    ]
    
    quantum_features = [
        'QUANTUM_SCORE', 'Enhanced_Sharpe', 'Alpha_Rank', 'Momentum_Strength',
        'Category_Consistency', 'All_Weather_Score', 'Regime_Adaptability',
        'Value_Factor', 'Diversification_Score', 'Cost_Efficiency'
    ]
    
    all_features = base_features + quantum_features
    available_features = [f for f in all_features if f in df.columns]
    
    if len(available_features) < 5:
        print("⚠️ Insufficient features for ML. Using rule-based scoring.")
        return None, available_features
    
    target_map = {
        '1Y': 'Absolute Returns - 1Y',
        '3Y': 'CAGR 3Y',
        '5Y': 'CAGR 5Y'
    }
    
    target_col = target_map.get(target_horizon, 'Absolute Returns - 1Y')
    if target_col not in df.columns:
        for alt_target in ['Absolute Returns - 1Y', 'CAGR 3Y', '3Y Avg Annual Rolling Return']:
            if alt_target in df.columns:
                target_col = alt_target
                break
    
    model_df = df[available_features + [target_col]].dropna()
    
    if len(model_df) < 100:
        print(f"⚠️ Only {len(model_df)} samples. Results may be less reliable.")
    
    X = model_df[available_features]
    y = model_df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=pd.qcut(y, q=5, labels=False, duplicates='drop')
    )
    
    print(f"📊 Training on {len(X_train)} samples, testing on {len(X_test)} samples")
    
    models = {}
    
    models['rf'] = RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, random_state=42, n_jobs=-1
    )
    
    models['gb'] = GradientBoostingRegressor(
        n_estimators=150, max_depth=7, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    
    models['et'] = ExtraTreesRegressor(
        n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
    )
    
    try:
        models['xgb'] = xgb.XGBRegressor(
            n_estimators=150, max_depth=7, learning_rate=0.05,
            subsample=0.8, random_state=42, verbosity=0
        )
    except:
        pass
    
    try:
        models['lgb'] = lgb.LGBMRegressor(
            n_estimators=150, max_depth=7, learning_rate=0.05,
            subsample=0.8, random_state=42, verbosity=-1
        )
    except:
        pass
    
    predictions = {}
    scores = {}
    
    for name, model in models.items():
        print(f"  Training {name.upper()}...", end='')
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        score = r2_score(y_test, pred)
        predictions[name] = pred
        scores[name] = score
        print(f" R²={score:.3f}")
    
    total_score = sum(scores.values())
    weights = {name: score/total_score for name, score in scores.items()}
    
    stacked_pred = sum(predictions[name] * weights[name] for name in predictions)
    stacked_r2 = r2_score(y_test, stacked_pred)
    stacked_rmse = np.sqrt(mean_squared_error(y_test, stacked_pred))
    
    print(f"\n🎯 STACKED MODEL PERFORMANCE:")
    print(f"  • R² Score: {stacked_r2:.3f}")
    print(f"  • RMSE: {stacked_rmse:.2f}%")
    print(f"  • Mean Absolute Error: {np.mean(np.abs(y_test - stacked_pred)):.2f}%")
    
    if 'rf' in models:
        importance = pd.DataFrame({
            'feature': available_features,
            'importance': models['rf'].feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("\n📈 TOP PREDICTIVE FEATURES:")
        for i, row in importance.head(5).iterrows():
            print(f"  {i+1}. {row['feature']}: {row['importance']:.3f}")
    
    def ensemble_predict(X_new):
        """Predicts using the stacked ensemble"""
        preds = {}
        for name, model in models.items():
            preds[name] = model.predict(X_new)
        result = sum(preds[name] * weights[name] for name in preds)
        return result
    
    ensemble = {
        'models': models,
        'weights': weights,
        'features': available_features,
        'performance': {'r2': stacked_r2, 'rmse': stacked_rmse}
    }
    
    return ensemble_predict, ensemble

## 4. Market Regime Detection

In [ ]:
def detect_market_regime_advanced(df):
    """Advanced market regime detection with multiple indicators"""
    
    print("\n🌐 ADVANCED MARKET REGIME DETECTION...")
    
    regime_signals = {}
    
    if 'Absolute Returns - 3M' in df.columns:
        recent_returns = df['Absolute Returns - 3M'].dropna()
        mean_return = recent_returns.mean()
        median_return = recent_returns.median()
        skewness = recent_returns.skew()
        
        regime_signals['mean_return'] = mean_return
        regime_signals['median_return'] = median_return
        regime_signals['skewness'] = skewness
        
        print(f"  • Mean 3M Return: {mean_return:.2f}%")
        print(f"  • Median 3M Return: {median_return:.2f}%")
        print(f"  • Skewness: {skewness:.2f}")
    
    if 'Volatility' in df.columns:
        avg_volatility = df['Volatility'].mean()
        regime_signals['volatility'] = avg_volatility
        print(f"  • Average Volatility: {avg_volatility:.2f}")
    
    if '% Equity Holding' in df.columns:
        equity_allocation = df['% Equity Holding'].mean()
        regime_signals['risk_appetite'] = equity_allocation
        print(f"  • Average Equity Allocation: {equity_allocation:.1f}%")
    
    if 'Absolute Returns - 1Y' in df.columns:
        positive_returns = (df['Absolute Returns - 1Y'] > 0).mean()
        regime_signals['market_breadth'] = positive_returns
        print(f"  • % Funds with Positive 1Y Returns: {positive_returns*100:.1f}%")
    
    regime_score = 0
    regime_factors = []
    
    if 'mean_return' in regime_signals:
        if regime_signals['mean_return'] > 5:
            regime_score += 2
            regime_factors.append("Strong Returns")
        elif regime_signals['mean_return'] > 0:
            regime_score += 1
            regime_factors.append("Positive Returns")
        elif regime_signals['mean_return'] < -5:
            regime_score -= 2
            regime_factors.append("Negative Returns")
        else:
            regime_score -= 1
            regime_factors.append("Weak Returns")
    
    if 'volatility' in regime_signals:
        if regime_signals['volatility'] < 10:
            regime_score += 1
            regime_factors.append("Low Volatility")
        elif regime_signals['volatility'] > 20:
            regime_score -= 1
            regime_factors.append("High Volatility")
    
    if 'market_breadth' in regime_signals:
        if regime_signals['market_breadth'] > 0.7:
            regime_score += 1
            regime_factors.append("Broad Participation")
        elif regime_signals['market_breadth'] < 0.3:
            regime_score -= 1
            regime_factors.append("Narrow Market")
    
    if regime_score >= 3:
        regime = "STRONG BULL"
        strategy = "Maximum Aggression - Small/Mid Caps, Sectoral, High Beta"
    elif regime_score >= 1:
        regime = "BULL"
        strategy = "Growth Focus - Flexi Cap, Balanced Advantage"
    elif regime_score >= -1:
        regime = "NEUTRAL"
        strategy = "Balanced - Multi Asset, Hybrid, Large Cap"
    elif regime_score >= -3:
        regime = "BEAR"
        strategy = "Defensive - Debt, Gold, Low Volatility Equity"
    else:
        regime = "STRONG BEAR"
        strategy = "Capital Preservation - Liquid, Overnight, Arbitrage"
    
    print(f"\n🎯 MARKET REGIME: {regime}")
    print(f"📊 Regime Score: {regime_score}")
    print(f"📌 Key Factors: {', '.join(regime_factors)}")
    print(f"💡 Recommended Strategy: {strategy}")
    
    return regime, regime_score, strategy

## 5. Portfolio Optimization

In [ ]:
def optimize_portfolio_allocation(df, selected_funds, risk_tolerance='balanced'):
    """Optimizes portfolio allocation using modern portfolio theory"""
    
    print("\n💼 OPTIMIZING PORTFOLIO ALLOCATION...")
    
    if len(selected_funds) < 2:
        print("  Need at least 2 funds for optimization")
        return {'equal_weight': 1.0 / len(selected_funds)}
    
    return_cols = ['Absolute Returns - 1Y', 'CAGR 3Y', '3Y Avg Annual Rolling Return']
    available_return_col = None
    
    for col in return_cols:
        if col in df.columns:
            available_return_col = col
            break
    
    if not available_return_col:
        print("  No return data available for optimization")
        return {fund: 1.0/len(selected_funds) for fund in selected_funds}
    
    portfolio_df = df[df['Name'].isin(selected_funds)]
    returns = portfolio_df.set_index('Name')[available_return_col].to_dict()
    volatilities = portfolio_df.set_index('Name')['Volatility'].to_dict() if 'Volatility' in df.columns else {}
    sharpe_ratios = portfolio_df.set_index('Name')['Sharpe Ratio'].to_dict() if 'Sharpe Ratio' in df.columns else {}
    
    risk_params = {
        'aggressive': {'target_return': 20, 'risk_weight': 0.3},
        'balanced': {'target_return': 12, 'risk_weight': 0.5},
        'conservative': {'target_return': 8, 'risk_weight': 0.7}
    }
    
    params = risk_params[risk_tolerance]
    n_funds = len(selected_funds)
    
    if sharpe_ratios and volatilities:
        scores = {}
        for fund in selected_funds:
            sharpe = sharpe_ratios.get(fund, 0.5)
            vol = volatilities.get(fund, 15)
            ret = returns.get(fund, 10)
            score = (sharpe * 0.4) + (ret / 100 * 0.3) + ((30 - vol) / 30 * 0.3)
            scores[fund] = max(score, 0.1)
        
        total_score = sum(scores.values())
        allocations = {fund: score/total_score for fund, score in scores.items()}
        
        if risk_tolerance == 'conservative':
            max_alloc = 0.30
        elif risk_tolerance == 'balanced':
            max_alloc = 0.40
        else:
            max_alloc = 0.50
        
        for fund in allocations:
            if allocations[fund] > max_alloc:
                excess = allocations[fund] - max_alloc
                allocations[fund] = max_alloc
                other_funds = [f for f in allocations if f != fund]
                for other in other_funds:
                    allocations[other] += excess / len(other_funds)
    else:
        allocations = {fund: 1.0/n_funds for fund in selected_funds}
    
    print("\n📊 OPTIMIZED ALLOCATION:")
    for fund, weight in sorted(allocations.items(), key=lambda x: x[1], reverse=True):
        print(f"  • {fund[:50]}: {weight*100:.1f}%")
    
    return allocations

## 6. Comprehensive Analysis Pipeline

In [ ]:
def execute_profit_maximization_analysis(df):
    """Main execution pipeline for maximum profit prediction"""
    
    print("\n" + "="*100)
    print(" "*35 + "🚀 PROFIT MAXIMIZATION ENGINE 🚀")
    print("="*100)
    
    print("\n📋 DATA PREPROCESSING...")
    
    numeric_cols = [col for col in df.columns if col not in ['Name', 'Sub Category', 'Plan', 'AMC', 
                                                              'Benchmark', 'Exit Load', 'Fund Manager',
                                                              'SIP Investment', 'SEBI Risk Category']]
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    initial_len = len(df)
    df = df[~df['Name'].str.contains('IDCW|Dividend|Direct', case=False, na=False)]
    print(f"  • Removed {initial_len - len(df)} duplicate plans")
    print(f"  • Analyzing {len(df)} unique funds")
    
    df = engineer_quantum_features(df)
    regime, regime_score, strategy = detect_market_regime_advanced(df)
    
    horizons = ['1Y', '3Y', '5Y']
    predictions = {}
    ensembles = {}
    
    for horizon in horizons:
        predictor, ensemble = build_stacked_ml_predictor(df, horizon)
        
        if predictor is not None:
            feature_df = df[ensemble['features']].fillna(0)
            df[f'PREDICTED_{horizon}_RETURN'] = predictor(feature_df)
            predictions[horizon] = df[f'PREDICTED_{horizon}_RETURN']
            ensembles[horizon] = ensemble
    
    if predictions:
        pred_cols = [f'PREDICTED_{h}_RETURN' for h in horizons if f'PREDICTED_{h}_RETURN' in df.columns]
        if pred_cols:
            weights = [0.5, 0.3, 0.2][:len(pred_cols)]
            df['MASTER_PREDICTION'] = sum(
                df[col] * w for col, w in zip(pred_cols, weights)
            )
    
    if 'MASTER_PREDICTION' in df.columns and 'Volatility' in df.columns:
        df['RISK_ADJUSTED_PREDICTION'] = df['MASTER_PREDICTION'] / (df['Volatility'] + 1)
    
    results = {}
    
    risk_profiles = {
        'conservative': {
            'volatility_cap': 8,
            'min_sharpe': 0.8,
            'categories': ['Debt:', 'Arbitrage', 'Conservative Hybrid', 'Liquid', 'Overnight']
        },
        'balanced': {
            'volatility_cap': 15,
            'min_sharpe': 0.6,
            'categories': ['Balanced', 'Large Cap', 'Multi Asset', 'Flexi Cap']
        },
        'aggressive': {
            'volatility_cap': 100,
            'min_sharpe': 0.4,
            'categories': ['Small Cap', 'Mid Cap', 'Sectoral', 'International', 'Thematic']
        }
    }
    
    print("\n" + "="*100)
    print(" "*35 + "💰 TOP RECOMMENDATIONS 💰")
    print("="*100)
    
    for risk_level, params in risk_profiles.items():
        print(f"\n{'='*80}")
        print(f"{risk_level.upper()} PORTFOLIO")
        print('='*80)
        
        filtered = df.copy()
        
        if 'Volatility' in filtered.columns:
            filtered = filtered[filtered['Volatility'] <= params['volatility_cap']]
        
        if 'Sharpe Ratio' in filtered.columns:
            filtered = filtered[filtered['Sharpe Ratio'] >= params['min_sharpe']]
        
        if 'Sub Category' in filtered.columns:
            category_mask = filtered['Sub Category'].apply(
                lambda x: any(cat in str(x) for cat in params['categories'])
            )
            filtered_category = filtered[category_mask]
            if len(filtered_category) > 0:
                filtered = filtered_category
        
        sort_col = 'MASTER_PREDICTION' if 'MASTER_PREDICTION' in filtered.columns else 'QUANTUM_SCORE'
        if sort_col in filtered.columns:
            filtered = filtered.sort_values(sort_col, ascending=False)
        
        top_funds = filtered.head(10)
        results[risk_level] = top_funds
        
        if not top_funds.empty:
            print("\n🏆 TOP 5 FUNDS:")
            for idx, (_, fund) in enumerate(top_funds.head(5).iterrows(), 1):
                print(f"\n{idx}. {fund['Name']}")
                print(f"   Category: {fund.get('Sub Category', 'N/A')}")
                if 'MASTER_PREDICTION' in fund.index:
                    print(f"   Predicted Return: {fund['MASTER_PREDICTION']:.2f}%")
                print(f"   Quantum Score: {fund.get('QUANTUM_SCORE', 0):.1f}/100")
                print(f"   Sharpe Ratio: {fund.get('Sharpe Ratio', 0):.2f}")
                print(f"   Expense Ratio: {fund.get('Expense Ratio', 0):.2f}%")
                print(f"   AUM: ₹{fund.get('AUM', 0):.0f} Cr")
        
        if len(top_funds) >= 3:
            selected_funds = top_funds.head(5)['Name'].tolist()
            allocations = optimize_portfolio_allocation(df, selected_funds, risk_level)
    
    print("\n" + "="*100)
    print(" "*35 + "🎯 SPECIAL RECOMMENDATIONS 🎯")
    print("="*100)
    
    if 'AUM' in df.columns and 'QUANTUM_SCORE' in df.columns:
        hidden_gems = df[(df['AUM'] < df['AUM'].quantile(0.3)) & 
                         (df['QUANTUM_SCORE'] > df['QUANTUM_SCORE'].quantile(0.7))]
        
        if not hidden_gems.empty:
            print("\n💎 HIDDEN GEMS (Low AUM, High Potential):")
            for _, fund in hidden_gems.head(3).iterrows():
                print(f"  • {fund['Name']}: Quantum Score {fund['QUANTUM_SCORE']:.1f}")
    
    if 'Category_Consistency' in df.columns:
        consistent = df[df['Category_Consistency'] >= 0.8].sort_values('QUANTUM_SCORE', ascending=False)
        
        if not consistent.empty:
            print("\n🎖️ MOST CONSISTENT PERFORMERS:")
            for _, fund in consistent.head(3).iterrows():
                print(f"  • {fund['Name']}: {fund['Category_Consistency']*100:.0f}% consistency")
    
    if 'Expense Ratio' in df.columns and 'Alpha' in df.columns:
        value_picks = df[(df['Expense Ratio'] < df['Expense Ratio'].quantile(0.25)) &
                         (df['Alpha'] > df['Alpha'].quantile(0.75))]
        
        if not value_picks.empty:
            print("\n💸 BEST VALUE (Low Cost, High Alpha):")
            for _, fund in value_picks.head(3).iterrows():
                print(f"  • {fund['Name']}: Alpha {fund['Alpha']:.2f}, Expense {fund['Expense Ratio']:.2f}%")
    
    print("\n" + "="*100)
    print(" "*35 + "📈 ACTION PLAN 📈")
    print("="*100)
    
    print(f"\nBASED ON CURRENT MARKET REGIME: {regime}")
    print(f"RECOMMENDED STRATEGY: {strategy}")
    
    print("\n🎯 IMMEDIATE ACTIONS:")
    
    if regime_score >= 1:
        print("  1. INCREASE equity allocation (60-80%)")
        print("  2. FOCUS on growth & momentum funds")
        print("  3. ADD small/mid cap exposure (20-30%)")
        print("  4. CONSIDER sectoral/thematic funds")
    elif regime_score >= -1:
        print("  1. MAINTAIN balanced allocation (50-50)")
        print("  2. FOCUS on multi-asset & hybrid funds")
        print("  3. KEEP large cap as core (40%)")
        print("  4. ADD arbitrage for stability")
    else:
        print("  1. REDUCE equity allocation (20-40%)")
        print("  2. INCREASE debt & liquid funds")
        print("  3. FOCUS on capital preservation")
        print("  4. CONSIDER gold allocation (10%)")
    
    print("\n💾 SAVING ANALYSIS...")
    
    output_file = 'profit_maximizer_results.xlsx'
    with pd.ExcelWriter(output_file) as writer:
        df.to_excel(writer, sheet_name='Full_Analysis', index=False)
        
        for risk_level, funds in results.items():
            if not funds.empty:
                funds[['Name', 'Sub Category', 'QUANTUM_SCORE', 'MASTER_PREDICTION', 
                       'Sharpe Ratio', 'Expense Ratio']].to_excel(
                    writer, sheet_name=f'Top_{risk_level}', index=False
                )
    
    print(f"✅ Results saved to {output_file}")
    
    print("\n" + "="*100)
    print(" "*30 + "🏁 ANALYSIS COMPLETE - GO MAKE MONEY! 🏁")
    print("="*100)
    
    return df, results

## 7. Visualization Dashboard

In [ ]:
def create_profit_dashboard(df):
    """Creates comprehensive visualization dashboard"""
    
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('🚀 MUTUAL FUND PROFIT MAXIMIZER DASHBOARD 🚀', fontsize=18, y=0.98)
    
    # 1. Predicted Returns Distribution
    ax1 = plt.subplot(3, 3, 1)
    if 'MASTER_PREDICTION' in df.columns:
        df['MASTER_PREDICTION'].hist(bins=30, ax=ax1, color='gold', edgecolor='black')
        ax1.axvline(df['MASTER_PREDICTION'].median(), color='red', linestyle='--', label='Median')
        ax1.set_xlabel('Predicted Return (%)')
        ax1.set_ylabel('Number of Funds')
        ax1.set_title('ML Predicted Returns Distribution')
        ax1.legend()
    
    # 2. Risk-Return Map
    ax2 = plt.subplot(3, 3, 2)
    if 'Volatility' in df.columns and 'MASTER_PREDICTION' in df.columns:
        scatter = ax2.scatter(df['Volatility'], df['MASTER_PREDICTION'], 
                             c=df['QUANTUM_SCORE'], s=30, alpha=0.6, cmap='RdYlGn')
        plt.colorbar(scatter, ax=ax2, label='Quantum Score')
        ax2.set_xlabel('Volatility (Risk)')
        ax2.set_ylabel('Predicted Return')
        ax2.set_title('Risk-Return Prediction Map')
        ax2.grid(True, alpha=0.3)
    
    # 3. Category Winners
    ax3 = plt.subplot(3, 3, 3)
    if 'Sub Category' in df.columns and 'QUANTUM_SCORE' in df.columns:
        top_categories = df.groupby('Sub Category')['QUANTUM_SCORE'].mean().sort_values(ascending=False).head(10)
        ax3.barh(range(len(top_categories)), top_categories.values, color='steelblue')
        ax3.set_yticks(range(len(top_categories)))
        ax3.set_yticklabels([cat[:20] for cat in top_categories.index], fontsize=8)
        ax3.set_xlabel('Avg Quantum Score')
        ax3.set_title('Top Categories by Quantum Score')
    
    # 4. Alpha Distribution
    ax4 = plt.subplot(3, 3, 4)
    if 'Alpha' in df.columns:
        positive_alpha = (df['Alpha'] > 0).sum()
        negative_alpha = (df['Alpha'] <= 0).sum()
        ax4.pie([positive_alpha, negative_alpha], labels=['Positive Alpha', 'Negative Alpha'],
                colors=['green', 'red'], autopct='%1.1f%%')
        ax4.set_title('Manager Skill Distribution')
    
    # 5. Expense Efficiency
    ax5 = plt.subplot(3, 3, 5)
    if 'Expense Ratio' in df.columns and 'Alpha' in df.columns:
        ax5.scatter(df['Expense Ratio'], df['Alpha'], alpha=0.5, color='purple')
        ax5.set_xlabel('Expense Ratio (%)')
        ax5.set_ylabel('Alpha')
        ax5.set_title('Cost vs Performance')
        ax5.grid(True, alpha=0.3)
        
        mask = df['Expense Ratio'].notna() & df['Alpha'].notna()
        if mask.sum() > 10:
            z = np.polyfit(df.loc[mask, 'Expense Ratio'], df.loc[mask, 'Alpha'], 1)
            p = np.poly1d(z)
            x_line = np.linspace(df['Expense Ratio'].min(), df['Expense Ratio'].max(), 100)
            ax5.plot(x_line, p(x_line), "r--", alpha=0.8, label='Trend')
            ax5.legend()
    
    # 6. Momentum Signal
    ax6 = plt.subplot(3, 3, 6)
    if 'Momentum_Strength' in df.columns:
        momentum_tiers = pd.qcut(df['Momentum_Strength'], q=5, 
                                 labels=['Very Weak', 'Weak', 'Neutral', 'Strong', 'Very Strong'])
        momentum_dist = momentum_tiers.value_counts()
        colors = ['darkred', 'red', 'yellow', 'lightgreen', 'green']
        ax6.bar(range(len(momentum_dist)), momentum_dist.values, color=colors)
        ax6.set_xticks(range(len(momentum_dist)))
        ax6.set_xticklabels(momentum_dist.index, rotation=45)
        ax6.set_ylabel('Number of Funds')
        ax6.set_title('Momentum Distribution')
    
    # 7. Sharpe Leaders
    ax7 = plt.subplot(3, 3, 7)
    if 'Sharpe Ratio' in df.columns:
        top_sharpe = df.nlargest(15, 'Sharpe Ratio')[['Name', 'Sharpe Ratio']]
        ax7.barh(range(len(top_sharpe)), top_sharpe['Sharpe Ratio'].values, color='teal')
        ax7.set_yticks(range(len(top_sharpe)))
        ax7.set_yticklabels([name[:25] for name in top_sharpe['Name']], fontsize=6)
        ax7.set_xlabel('Sharpe Ratio')
        ax7.set_title('Top Risk-Adjusted Performers')
    
    # 8. AUM vs Performance
    ax8 = plt.subplot(3, 3, 8)
    if 'AUM' in df.columns and 'QUANTUM_SCORE' in df.columns:
        ax8.scatter(np.log1p(df['AUM']), df['QUANTUM_SCORE'], alpha=0.5, color='orange')
        ax8.set_xlabel('Log(AUM)')
        ax8.set_ylabel('Quantum Score')
        ax8.set_title('Size vs Quality')
        ax8.grid(True, alpha=0.3)
    
    # 9. Risk Category Performance
    ax9 = plt.subplot(3, 3, 9)
    if 'SEBI Risk Category' in df.columns and 'MASTER_PREDICTION' in df.columns:
        risk_perf = df.groupby('SEBI Risk Category')['MASTER_PREDICTION'].mean().sort_values()
        ax9.bar(range(len(risk_perf)), risk_perf.values, 
                color=['green', 'yellow', 'orange', 'red', 'darkred'][:len(risk_perf)])
        ax9.set_xticks(range(len(risk_perf)))
        ax9.set_xticklabels(risk_perf.index, rotation=45, ha='right')
        ax9.set_ylabel('Avg Predicted Return (%)')
        ax9.set_title('Risk Category Returns')
    
    plt.tight_layout()
    plt.savefig('profit_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("📊 Dashboard saved as 'profit_dashboard.png'")

## 8. Main Execution Function

In [ ]:
def maximize_profits(csv_path):
    """
    THE MONEY MAKER - Just pass your CSV and watch the magic
    
    Parameters:
    -----------
    csv_path : str or DataFrame
        Path to CSV file or pandas DataFrame
    
    Returns:
    --------
    results_df : DataFrame
        Full analysis with predictions and scores
    recommendations : dict
        Top fund recommendations by risk profile
    
    Usage:
    ------
    df = pd.read_csv('mutual_funds.csv')
    results_df, recommendations = maximize_profits(df)
    """
    
    if isinstance(csv_path, str):
        print(f"📂 Loading data from {csv_path}...")
        df = pd.read_csv(csv_path)
    else:
        df = csv_path
    
    print(f"✅ Loaded {len(df)} funds with {len(df.columns)} features")
    
    results_df, recommendations = execute_profit_maximization_analysis(df)
    create_profit_dashboard(results_df)
    
    print("\n" + "💰"*50)
    print(" "*15 + "PROFIT MAXIMIZATION COMPLETE!")
    print(" "*10 + "Check 'profit_maximizer_results.xlsx' for full results")
    print(" "*10 + "Dashboard saved as 'profit_dashboard.png'")
    print("💰"*50 + "\n")
    
    return results_df, recommendations

## 9. Example Usage

Run the cells below with your own mutual fund data!

In [ ]:
# Example 1: Load from CSV file
# results_df, recommendations = maximize_profits('your_mutual_funds.csv')

# Example 2: Load DataFrame directly
# df = pd.read_csv('your_mutual_funds.csv')
# results_df, recommendations = maximize_profits(df)

print("""
╔══════════════════════════════════════════════════════════════╗
║          💰 ULTIMATE PROFIT MAXIMIZER READY! 💰              ║
║                                                              ║
║  To use:                                                     ║
║  1. df = pd.read_csv('your_mutual_funds.csv')              ║
║  2. results, recommendations = maximize_profits(df)         ║
║                                                              ║
║  That's it! NO BULLSHIT, JUST PROFITS! 🚀                  ║
╚══════════════════════════════════════════════════════════════╝
""")

## 10. Explore Results

After running the analysis, explore your results:

In [ ]:
# View top conservative picks
# recommendations['conservative'].head()

# View top aggressive picks
# recommendations['aggressive'].head()

# View top balanced picks
# recommendations['balanced'].head()

# View funds with highest quantum score
# results_df.nlargest(10, 'QUANTUM_SCORE')[['Name', 'QUANTUM_SCORE', 'MASTER_PREDICTION', 'Sharpe Ratio']]

# View funds with highest predicted returns
# results_df.nlargest(10, 'MASTER_PREDICTION')[['Name', 'MASTER_PREDICTION', 'QUANTUM_SCORE', 'Volatility']]